# City GPT
Character-level Transformer trained on GeoNames allCountries data.

**Settings → Accelerator → GPU T4 x2** before running.

Data is downloaded to Kaggle's ephemeral `/kaggle/working/` storage.  
Model checkpoints are saved there too — they persist across sessions (up to 20 GB).

In [ ]:
# ── 1. Install uv and clone the repo ───────────────────────────────────────
import os, subprocess, sys

subprocess.run(["curl", "-LsSf", "https://astral.sh/uv/install.sh"],
               stdout=open("/tmp/uv_install.sh", "w"), check=True)
subprocess.run(["sh", "/tmp/uv_install.sh"], check=True)
os.environ["PATH"] += ":/root/.local/bin"

WORKDIR = "/kaggle/working/city-gpt"
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "https://github.com/igui/city-gpt.git", WORKDIR], check=True)
else:
    subprocess.run(["git", "-C", WORKDIR, "pull"], check=True)

os.chdir(WORKDIR)
subprocess.run(["uv", "sync", "--quiet"], check=True)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── 2. Configure paths ─────────────────────────────────────────────────────
# All data and the checkpoint live under /kaggle/working/ which persists
# across Kaggle sessions (up to 20 GB output quota).
import sys
sys.path.insert(0, WORKDIR)

if 'city_gpt' in sys.modules:
    del sys.modules['city_gpt']

import city_gpt

city_gpt.DATA_DIR   = f"{WORKDIR}/data"
city_gpt.ZIP_PATH   = f"{city_gpt.DATA_DIR}/allCountries.zip"
city_gpt.RAW_PATH   = f"{city_gpt.DATA_DIR}/allCountries.txt"
city_gpt.DB_PATH    = f"{city_gpt.DATA_DIR}/corpus.db"
city_gpt.MODEL_PATH = f"{WORKDIR}/city_gpt.pt"

os.makedirs(city_gpt.DATA_DIR, exist_ok=True)

print(f"DATA_DIR   : {city_gpt.DATA_DIR}")
print(f"MODEL_PATH : {city_gpt.MODEL_PATH}")
print(f"device     : {city_gpt.device}")

In [ ]:
# ── 3. Prepare data ────────────────────────────────────────────────────────
# Downloads allCountries.zip (~1.5 GB) and builds corpus.db.
# Skip this cell if corpus.db already exists from a previous session.
if os.path.exists(city_gpt.DB_PATH):
    print(f"corpus.db already exists at {city_gpt.DB_PATH}, skipping prepare.")
    print("Delete it and re-run this cell to rebuild from scratch.")
else:
    import argparse
    args = argparse.Namespace(max_rows=None)   # set e.g. max_rows=500_000 for a quick test
    city_gpt.cmd_prepare(args)

In [ ]:
# ── 4. Train ───────────────────────────────────────────────────────────────
# Checkpoint is saved every 100 steps to MODEL_PATH.
# If a checkpoint already exists, training starts fresh (new weights).
# To resume from a checkpoint, load it manually first:
#
#   import torch
#   ckpt  = torch.load(city_gpt.MODEL_PATH, map_location=city_gpt.device)
#   model = city_gpt.GPTLanguageModel(len(ckpt['stoi'])).to(city_gpt.device)
#   model.load_state_dict(ckpt['model_state'])
#
import argparse
city_gpt.cmd_train(argparse.Namespace())

In [ ]:
# ── 5. Generate samples ────────────────────────────────────────────────────
import argparse
args = argparse.Namespace(prompt="", n=500, temperature=1.0, top_k=None)
city_gpt.cmd_generate(args)

In [ ]:
# ── 6. Generate with a prompt ──────────────────────────────────────────────
# Format: <flag><class-emoji><feature-code>|
# Examples:
#   🇺🇸🏙️PPL|   US populated place
#   🇯🇵⛰️MT|    Japanese mountain
#   🇩🇪💧LK|    German lake
#   🇬🇧🏛️ADM1|  UK administrative region
import argparse
args = argparse.Namespace(
    prompt="🇺🇸🏙️PPL|",
    n=200,
    temperature=0.8,
    top_k=50,
)
city_gpt.cmd_generate(args)